# Electricity Master Workflow
## Trustworthy Time-Series Foundation Model Evaluation — Energy

**Primary series:** South Australian electricity demand, T4, half-hourly  
**Authoritative test:** 46,176 observations / 962 days  
**Protocol A:** rolling one-step, 30-minute horizon  
**Protocol B:** true 48-step / 24-hour day-ahead

This safe orchestration notebook complements—without replacing—the detailed Electricity notebooks.

## Table of Contents

1. Research Objective
2. Configuration and Safety Controls
3. Environment and Paths
4. Artifact Verification
5. Dataset Overview
6. Train / Validation / Test Design
7. Forecast Protocols
8. Model Inventory
9. Optional Forecast Generation
10. Load Protocol A Artifact
11. Protocol A Accuracy
12. Protocol B Artifact
13. Protocol B Accuracy
14. Protocol Comparison
15. Horizon-Specific Behaviour
16. Regime-Conditional Robustness
17. Temporal Stability
18. Uncertainty Calibration
19. Statistical Significance
20. Trustworthiness Evidence
21. Final Electricity Findings
22. Detailed Notebook Links

## Workflow

```text
Raw T4 overview (optional)
          |
          v
Frozen Protocol A vectors ----\
                               +--> validation --> metrics and evidence --> conclusions
Frozen Protocol B vectors ----/
          |
          +--> protected horizon, regime, stability, uncertainty, DM, and trust tables
```

**Reproducibility notice:** Safe mode loads artifacts only. It never imports TensorFlow, Chronos, TimesFM, or other model frameworks; trains models; regenerates forecasts; or writes authoritative results.

## 1. Research Objective

The study evaluates Point Forecast Accuracy, Regime-Conditional Robustness, Temporal Stability, Uncertainty Calibration, Transparency and Auditability, Statistical Significance and practical effects, followed by a secondary Exploratory Composite Trustworthiness Summary.

## 2. Configuration and Safety Controls

All expensive and overwrite controls default to `False`. An explicit confirmation token is required even to enter generation guidance; authoritative overwrite remains unsupported here.

In [ ]:
RUN_EXPENSIVE_MODELS = False
OVERWRITE_AUTHORITATIVE_ARTIFACTS = False
VERIFY_ARTIFACTS = True
LOAD_RAW_DATA_OVERVIEW = True
EXPENSIVE_CONFIRMATION = ""  # must equal "I UNDERSTAND THIS MAY REGENERATE FORECASTS"

SAFETY = {
    "RUN_EXPENSIVE_MODELS": RUN_EXPENSIVE_MODELS,
    "OVERWRITE_AUTHORITATIVE_ARTIFACTS": OVERWRITE_AUTHORITATIVE_ARTIFACTS,
    "VERIFY_ARTIFACTS": VERIFY_ARTIFACTS,
    "LOAD_RAW_DATA_OVERVIEW": LOAD_RAW_DATA_OVERVIEW,
}
SAFETY

## 3. Environment and Paths

Repository discovery works from the repository root or any nested notebook directory.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found from the current working directory.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
ELECTRICITY_RESULTS_DIR = RESULTS_DIR / "electricity"
FIGURES_DIR = ROOT / "figures"
NOTEBOOKS_DIR = ROOT / "notebooks"

from src.metrics import mae, rmse, mape, smape

pd.DataFrame({"Path": [ROOT, DATA_DIR, RESULTS_DIR, ELECTRICITY_RESULTS_DIR, FIGURES_DIR]}, index=["ROOT", "DATA_DIR", "RESULTS_DIR", "ELECTRICITY_RESULTS_DIR", "FIGURES_DIR"])

## 4. Artifact Verification

The repository verifier is artifact-only and checks all protected hashes plus forecast schemas, keys, row counts, and reproduced metrics.

In [ ]:
verification_summary = "SKIPPED"
if VERIFY_ARTIFACTS:
    completed = subprocess.run(
        [sys.executable, str(ROOT / "src" / "verify_research_artifacts.py")],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    verification_summary = lines[-1] if lines else completed.stderr.strip()
    if completed.returncode != 0:
        raise RuntimeError(verification_summary)
display(pd.DataFrame([{"Artifact verification": verification_summary}]))

## 5. Dataset Overview

The lightweight parser reads only the T4 line from the TSF source. If it is absent, the notebook continues with frozen forecast artifacts.

In [ ]:
tsf_path = DATA_DIR / "electricity" / "australian_electricity_demand_dataset.tsf"
sa = None
if LOAD_RAW_DATA_OVERVIEW and tsf_path.is_file():
    with tsf_path.open("r", encoding="utf-8") as stream:
        t4_line = next((line.strip() for line in stream if line.startswith("T4:")), None)
    if t4_line is None:
        raise ValueError("T4 series was not found in the TSF source.")
    series_name, state, start_text, values_text = t4_line.split(":", 3)
    values = np.fromstring(values_text, sep=",")
    start = pd.to_datetime(start_text, format="%Y-%m-%d %H-%M-%S")
    index = pd.date_range(start, periods=len(values), freq="30min")
    sa = pd.Series(values, index=index, name="Demand")
    overview = pd.DataFrame([{
        "Series": series_name, "State": state, "Observations": len(sa),
        "Start": sa.index.min(), "End": sa.index.max(), "Frequency": "30 minutes",
        "Missing values": int(sa.isna().sum()), "Duplicate timestamps": int(sa.index.duplicated().sum()),
    }])
    display(overview); display(sa.describe().to_frame())
elif not tsf_path.is_file():
    display(Markdown("**Raw electricity source data is not available locally. Continuing with frozen authoritative forecast artifacts.**"))
else:
    display(Markdown("Raw-data overview disabled; continuing with frozen authoritative forecast artifacts."))

## 6. Train / Validation / Test Design

All partitions are chronological, contiguous, non-overlapping, and aligned to complete days.

In [ ]:
partitions = pd.DataFrame([
    ("Development train", "2002-01-01 00:00", "2011-06-23 23:30", 166128, 3461),
    ("Internal validation", "2011-06-24 00:00", "2012-07-12 23:30", 18480, 385),
    ("Frozen final test", "2012-07-13 00:00", "2015-03-01 23:30", 46176, 962),
], columns=["Partition", "Start", "End", "Observations", "Days"])
display(partitions)

## 7. Forecast Protocols

| Property | Protocol A | Protocol B |
|---|---|---|
| Horizon | One 30-minute step | 48 half-hours / 24 hours |
| Origins | Every test timestamp | 962 non-overlapping midnights |
| Updates | Actual at t only after forecast(t) | No actual inside the forecast day |
| Keys | Timestamp | Origin, Horizon, Timestamp |

```text
A: history --> forecast(t) --> reveal actual(t) --> next forecast
B: history at midnight --> forecast h=1...48 with no within-day updates
```

## 8. Model Inventory

In [ ]:
model_inventory = pd.DataFrame([
    ("Naive", "Baseline"), ("Daily Seasonal Naive", "Baseline"),
    ("Weekly Seasonal Naive", "Baseline"), ("Moving Average", "Baseline"),
    ("DHR-ARIMA", "Statistical"), ("LSTM", "Deep Learning"),
    ("Chronos-Bolt-Tiny", "Foundation Model — zero-shot"),
    ("TimesFM", "Foundation Model — zero-shot"),
], columns=["Model", "Class"])
display(model_inventory)

## 9. OPTIONAL — EXPENSIVE / RESEARCH REGENERATION

Generation remains in [11 — Baselines](electricity/11_Electricity_Baselines.ipynb), [11b — DHR-ARIMA](electricity/11b_Electricity_Statistical_Model.ipynb), [12 — LSTM](electricity/12_Electricity_LSTM.ipynb), and [13 — Foundation Models](electricity/13_Electricity_Foundation_Models.ipynb). Safe `Run All` never executes them.

In [ ]:
if RUN_EXPENSIVE_MODELS:
    required = "I UNDERSTAND THIS MAY REGENERATE FORECASTS"
    if EXPENSIVE_CONFIRMATION != required:
        raise PermissionError("Expensive mode requires the exact confirmation token.")
    if OVERWRITE_AUTHORITATIVE_ARTIFACTS:
        raise PermissionError("Authoritative overwrite is intentionally unsupported in the master notebook.")
    display(Markdown("Use the linked generation notebooks after reviewing their model-specific controls."))
else:
    display(Markdown("**SAFE MODE:** expensive generation skipped; no heavy model package was imported."))

## 10. Load Protocol A Artifact

Protocol A must contain 46,176 aligned, complete, unique, sorted timestamps.

In [ ]:
a_path = ELECTRICITY_RESULTS_DIR / "protocol_a_validated_forecasts.csv"
a = pd.read_csv(a_path, parse_dates=["Timestamp"])
model_columns = ["Naive", "Daily_Seasonal_Naive", "Weekly_Seasonal_Naive", "Moving_Average", "DHR_ARIMA", "LSTM", "Chronos_Bolt_Tiny", "TimesFM"]
a_checks = {
    "46,176 rows": len(a) == 46176, "unique timestamps": a.Timestamp.is_unique,
    "sorted timestamps": a.Timestamp.is_monotonic_increasing,
    "no missing values": not a[["Actual", *model_columns]].isna().any().any(),
}
assert all(a_checks.values()), a_checks
display(pd.DataFrame({"Check": a_checks.keys(), "PASS": a_checks.values()}))

## 11. Protocol A Accuracy

MAE, RMSE, MAPE, and sMAPE are recomputed from the vector. Protected MASE-48 values are joined from the authoritative cross-domain metric table because the denominator belongs to the frozen pre-test series.

In [ ]:
display_names = {c: c.replace("_", " ").replace("DHR ARIMA", "DHR-ARIMA").replace("Chronos Bolt Tiny", "Chronos-Bolt-Tiny") for c in model_columns}
rows=[]
for c in model_columns:
    rows.append({"Model": display_names[c], "MAE": mae(a.Actual,a[c]), "RMSE": rmse(a.Actual,a[c]), "MAPE": mape(a.Actual,a[c]), "sMAPE": smape(a.Actual,a[c])})
a_metrics=pd.DataFrame(rows)
cross=pd.read_csv(RESULTS_DIR/"cross_domain_model_comparison.csv")
a_mase=cross[cross.Protocol.eq("Protocol A: rolling one-step 30-minute")][["Model","MASE_48"]]
a_metrics=a_metrics.merge(a_mase,on="Model",how="left").sort_values("MASE_48").reset_index(drop=True)
a_metrics.insert(0,"Rank",np.arange(1,len(a_metrics)+1))
display(a_metrics.style.format({c:"{:.4f}" for c in ["MAE","RMSE","MAPE","sMAPE","MASE_48"]}))

## 12. Protocol B Artifact

Protocol B must contain 962 origins × 48 horizons, with unique origin/horizon keys and no within-window updates.

In [ ]:
b_path = ELECTRICITY_RESULTS_DIR / "protocol_b_validated_forecasts.csv"
b = pd.read_csv(b_path, parse_dates=["Origin", "Timestamp"])
b_checks = {
    "46,176 rows": len(b) == 46176, "962 origins": b.Origin.nunique() == 962,
    "48 horizons per origin": b.groupby("Origin").Horizon.count().eq(48).all(),
    "unique origin/horizon": not b.duplicated(["Origin", "Horizon"]).any(),
    "no missing values": not b[["Actual", *model_columns]].isna().any().any(),
}
assert all(b_checks.values()), b_checks
display(pd.DataFrame({"Check": b_checks.keys(), "PASS": b_checks.values()}))

## 13. Protocol B Accuracy

In [ ]:
rows=[]
for c in model_columns:
    rows.append({"Model": display_names[c], "MAE": mae(b.Actual,b[c]), "RMSE": rmse(b.Actual,b[c]), "MAPE": mape(b.Actual,b[c]), "sMAPE": smape(b.Actual,b[c])})
b_metrics=pd.DataFrame(rows)
b_mase=cross[cross.Protocol.eq("Protocol B: 48-step day-ahead")][["Model","MASE_48"]]
b_metrics=b_metrics.merge(b_mase,on="Model",how="left").sort_values("MASE_48").reset_index(drop=True)
b_metrics.insert(0,"Rank",np.arange(1,len(b_metrics)+1))
display(b_metrics.style.format({c:"{:.4f}" for c in ["MAE","RMSE","MAPE","sMAPE","MASE_48"]}))

## 14. Protocol Comparison

Ranks are compared without merging the protocols. The reversal of DHR-ARIMA and the strengthened seasonal baseline expose horizon and information-set dependence.

In [ ]:
rank_comparison = a_metrics[["Model","Rank","MASE_48"]].merge(b_metrics[["Model","Rank","MASE_48"]],on="Model",suffixes=("_A","_B"))
rank_comparison["Rank change B minus A"] = rank_comparison["Rank_B"] - rank_comparison["Rank_A"]
display(rank_comparison.sort_values("Rank_A"))

## 15. Horizon-Specific Behaviour

The protected horizon table is loaded directly; forecasts are not regenerated.

In [ ]:
horizon = pd.read_csv(ELECTRICITY_RESULTS_DIR / "protocol_b_validated_horizon_metrics.csv")
selected = horizon[horizon.Model.isin(["TimesFM", "Chronos_Bolt_Tiny", "Daily_Seasonal_Naive", "DHR_ARIMA"])].copy()
selected["Model"] = selected["Model"].map(display_names)
pivot = selected.pivot(index="Horizon", columns="Model", values="MASE_48")
display(pivot.loc[[1, 12, 24, 36, 48]].round(4))
ax=pivot.plot(figsize=(10,4)); ax.set(title="Protocol B MASE-48 by horizon",ylabel="MASE-48"); plt.tight_layout(); plt.show()

## 16. Regime-Conditional Robustness

These are predefined conditional demand/variability slices—not adversarial robustness.

In [ ]:
a_rob=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_a_robustness.csv")
b_rob=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_b_robustness.csv")
rob_summary=pd.concat([a_rob.assign(Protocol_Label="A"),b_rob.assign(Protocol_Label="B")],ignore_index=True)
display(rob_summary.groupby(["Protocol_Label","Model"],as_index=False).agg(Regimes=("Regime","nunique"),Mean_MASE_48=("MASE_48","mean")).sort_values(["Protocol_Label","Mean_MASE_48"]).round(4))

## 17. Temporal Stability

Earlier, Middle, and Later segments measure performance stability over the test period—not geographic, cross-dataset, OOD, or structural-break generalisation.

In [ ]:
a_stab=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_a_generalisation.csv")
b_stab=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_b_generalisation.csv")
display(Markdown("**Protocol A segment MASE-48**")); display(a_stab.pivot(index="Model",columns="Segment",values="MASE_48").round(4))
display(Markdown("**Protocol B segment MASE-48**")); display(b_stab.pivot(index="Model",columns="Segment",values="MASE_48").round(4))

## 18. Uncertainty Calibration

Chronos has lower absolute error from nominal 80% marginal coverage under both protocols. This is not universal calibration superiority; interval width and sharpness remain relevant.

In [ ]:
uncertainty=pd.read_csv(ELECTRICITY_RESULTS_DIR/"uncertainty_summary.csv")
display(uncertainty[["Protocol","Model","Interval","Nominal_Coverage","Empirical_Coverage","Average_Width","Available"]])

## 19. Statistical Significance

Protocol A uses timestamp-level loss differentials with HAC dependence adjustment. Protocol B aggregates loss by daily origin before HAC inference. Benjamini–Hochberg correction controls the reported comparison family, and loss-function sensitivity limits unconditional claims.

In [ ]:
a_dm=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_a_dm_tests.csv")
b_dm=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_b_dm_tests.csv")
key_models={"TimesFM","Chronos_Bolt_Tiny","DHR_ARIMA","Daily_Seasonal_Naive"}
def concise_dm(frame):
    keep=frame.Model_1.isin(key_models)&frame.Model_2.isin(key_models)
    return frame.loc[keep,["Loss","Model_1","Model_2","DM_Statistic","p_value_BH","Lower_Error_Winner","Significant_BH_0.05"]]
display(Markdown("**Protocol A key comparisons**")); display(concise_dm(a_dm))
display(Markdown("**Protocol B key comparisons**")); display(concise_dm(b_dm))

## 20. Trustworthiness Evidence

Dimension-level evidence is primary. Composite scores are secondary, researcher-defined, comparison-set dependent summaries and must not dominate interpretation.

In [ ]:
trust_cols=["Model","Relative Accuracy Score","Relative Robustness Score","Relative Generalisation Score","Uncertainty Score","Explainability Score"]
composite_cols=["Model","Overall Trust Score - Missing Evidence Penalised","Evidence-Available Trust Score","Unavailable Dimensions"]
a_trust=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_a_trust_scores.csv")
b_trust=pd.read_csv(ELECTRICITY_RESULTS_DIR/"protocol_b_trust_scores.csv")
display(Markdown("**Protocol A component evidence**")); display(a_trust[trust_cols].rename(columns={"Relative Generalisation Score":"Relative Temporal Stability Score"}))
display(Markdown("**Protocol B component evidence**")); display(b_trust[trust_cols].rename(columns={"Relative Generalisation Score":"Relative Temporal Stability Score"}))
display(Markdown("**Secondary exploratory composites — Protocol A**")); display(a_trust[composite_cols])
display(Markdown("**Secondary exploratory composites — Protocol B**")); display(b_trust[composite_cols])

## 21. Final Electricity Findings

- TimesFM leads both protocols.
- DHR-ARIMA is excellent one-step but weak day-ahead.
- Daily Seasonal Naive is a strong day-ahead comparator.
- Chronos is second day-ahead.
- Chronos has lower absolute 80% marginal coverage error than TimesFM.
- Model ranking is strongly protocol- and horizon-dependent.

## 22. Detailed Notebook Links

- [10 — Electricity EDA](electricity/10_Electricity_EDA.ipynb)
- [11 — Baselines](electricity/11_Electricity_Baselines.ipynb)
- [11b — Statistical Model](electricity/11b_Electricity_Statistical_Model.ipynb)
- [12 — LSTM](electricity/12_Electricity_LSTM.ipynb)
- [13 — Foundation Models](electricity/13_Electricity_Foundation_Models.ipynb)
- [14 — Model Validation Audit](electricity/14_Electricity_Model_Validation_Audit.ipynb)
- [15 — Trustworthiness Evidence](electricity/15_Electricity_Trustworthiness_Evidence.ipynb)
- [16 — Trustworthiness Composite](electricity/16_Electricity_Trustworthiness.ipynb)
- [17 — Statistical Significance](electricity/17_Electricity_Statistical_Significance.ipynb)